# EXP-2026-001 on Colab: Stage 5 pilot, part 1 of 2 (training and previews)

This notebook trains the MedSegFactory recipe with the pre-op adaptation for 1,000 steps on all accepted training patients. The text is the class terms of each slice (no size, no resection wording). The plan was fixed in `configs/experiments/EXP-2026-001.json` (block `stage5_pilot`) before this run.

- **Training:** train seed 2026, microbatch 4 x accumulation 5, bf16, lr 1e-5.
- **Previews at steps 0, 500 and 1000:** the 16 Stage 4 training slices and 12 validation slices chosen from pre-op data only. Each is generated from the noised pre-op pair (`pair_preop`) and from pure noise (`pure_noise`). Step 0 shows what the MSF checkpoint does before any RHUH training.
- **Wrong-patient control at step 1000:** validation slices get another patient's pre-op pair and pre-op channels, with their own text.
- **Fixed-noise loss:** on the training and validation previews, at t = 100 to 981.

The validation previews use post-op latents only as the loss target. The class text comes from the post-op label, so it is an oracle condition. The test set is not touched.

Expect about 75 minutes. Everything is written to Colab's local disk first, then copied to `MyDrive/EXP-2026-001_stage5/<RUN_TAG>/` in this account's own Drive after each preview and at the end, including the 3.5 GiB step-1000 UNet that part 2 needs. If a copy to Drive fails, training continues. The last cell forces the upload to finish, reconnects Drive and confirms every file, and Colab may ask you to approve the reconnection. The PNG grids show patient images, so keep them private. If the session dies, rerun everything with a new `RUN_TAG`.

**Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "exp-2026-001-implementation"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
WORK = "/content/MedSegFactory"
SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MSF_REPO, MSF_FILE = "JohnWeck/StableDiffusion", "checkpoint-300.pth"
RUN_TAG = "r2"  # use a new tag for every attempt; a run folder is never reused (r1 lost its folder on Drive)
OUTPUT_ROOT = "/content/drive/MyDrive/EXP-2026-001_stage5"  # this account's own Drive, not through the shared folder's shortcut

## 1. Mount Drive, get the latest code from the fork, link the data

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
PRIVATE = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
assert (PRIVATE / "split/split.json").exists(), "The synthesis folder is missing on Drive"
assert any((PRIVATE / "export_v1/val").glob("*.npz")), "export_v1/val is missing on Drive: copy it from the laptop folder"
RUN = Path(OUTPUT_ROOT) / RUN_TAG
assert not RUN.exists(), f"{RUN} exists: choose a new RUN_TAG (records are never overwritten)"
LOCAL_RUN = Path(f"/content/stage5_{RUN_TAG}")  # written first, then copied to RUN on Drive
usage = shutil.disk_usage("/content/drive/MyDrive")
print(f"Free space on Drive: {usage.free / 2**30:.1f} GiB")
assert not usage.total or usage.free > 6 * 2**30, "Stage 5 needs about 6 GiB free on Drive: free some space first"
assert not LOCAL_RUN.exists(), f"{LOCAL_RUN} exists: choose a new RUN_TAG"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
                "nibabel>=5.3,<6", "SimpleITK==2.5.6", "pydicom>=3,<4", "scikit-image>=0.25"], check=True)
commit = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(commit)
STARTED = time.perf_counter()

## 2. Models: MedSegFactory checkpoint (strict), frozen VAE and CLIP, schedulers

In [ ]:
import numpy as np
import torch
from diffusers import AutoencoderKL, DDPMScheduler
from huggingface_hub import hf_hub_download
from transformers import CLIPTextModel, CLIPTokenizer

from rhuh_synthesis.manifest import file_sha256
from rhuh_synthesis.model import reference_ddim_scheduler
from rhuh_synthesis.provenance import code_version, config_sha256, environment
from rhuh_synthesis.stage3 import load_reference_unet
from StableDiffusion.Our_UNet import UNet2DConditionModel

config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))
S4, S5 = config["stage4_overfit"], config["stage5_pilot"]
DIAG = config["stage4_r2_diagnostics"]  # metric definitions
TRAIN, PREVIEWS = S5["training"], S5["previews"]
CODEC = config["representation"]["mask_codec"]
checkpoint = hf_hub_download(MSF_REPO, MSF_FILE)
unet_config = UNet2DConditionModel.load_config(SD_REPO, subfolder="unet")
unet4, load_report = load_reference_unet(UNet2DConditionModel, unet_config, checkpoint)
assert unet4 is not None, "Checkpoint keys do not match the vendored UNet: stop and report"
vae = AutoencoderKL.from_pretrained(SD_REPO, subfolder="vae").to("cuda").eval().requires_grad_(False)
tokenizer = CLIPTokenizer.from_pretrained(SD_REPO, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(SD_REPO, subfolder="text_encoder").to("cuda").eval().requires_grad_(False)
noise_scheduler = DDPMScheduler.from_pretrained(SD_REPO, subfolder="scheduler")
ddim = reference_ddim_scheduler()
record = {
    "stage": "5_pilot_train", "run_tag": RUN_TAG, "commit": commit, "code_version": code_version(WORK),
    "config_sha256": config_sha256(config), "stage5_config": S5, "environment": environment(),
    "frozen_blocks": {name: config[name] for name in ("text", "representation", "evaluation", "training", "sampling", "stage5_pilot")},
    "torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
    "checkpoint": {"repo": MSF_REPO, "file": MSF_FILE, "sha256": file_sha256(checkpoint),
                   "missing_keys": len(load_report["missing_keys"]), "unexpected_keys": len(load_report["unexpected_keys"])},
}
print({k: record[k] for k in ("commit", "config_sha256", "torch", "cuda", "gpu")}, record["checkpoint"])

## 3. Preflight: adapter equivalence with TF32 off (and on, for reference)

In [ ]:
from rhuh_synthesis.stage3 import adapter_equivalence

limits = S4["preflight_adapter_equivalence"]
record["equivalence_tf32_off"] = adapter_equivalence(unet4, latent_hw=(64, 64), cross_dim=768, device="cuda", tf32=False)
record["equivalence_tf32_on"] = adapter_equivalence(unet4, latent_hw=(64, 64), cross_dim=768, device="cuda", tf32=True)
off = record["equivalence_tf32_off"]
record["preflight_pass"] = bool(off["relative_difference"] <= limits["relative_difference_max"]
                                and off["copied_weights_identical"] and off["added_weights_zero"])
print("TF32 off:", off)
print("TF32 on: ", record["equivalence_tf32_on"])
assert record["preflight_pass"], "Adapter preflight failed: stop here and send the two lines above"


## 4. Training patients, preview slices and the feature caches (about 15 minutes)

In [ ]:
from rhuh_synthesis.overfit import (image_start_latents, label_volumes, mask_latents, ranked_patients, restrict,
                                   select_overfit_set)
from rhuh_synthesis.pilot import accepted_patients, local_copies, select_preview_slices, training_cache
from rhuh_synthesis.representation import crop_forward
from rhuh_synthesis.stage3 import clip_encoder
from rhuh_synthesis.training import build_feature_cache
from rhuh_synthesis.working_grid import to_working_grid

split = json.loads((PRIVATE / "split/split.json").read_text(encoding="utf-8"))
policy = json.loads((PRIVATE / "registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
train_patients = [p for p in split["roles"]["train"]["patients"] if p not in policy["excluded_training"]]
paths = {p: next(PRIVATE.glob(f"export_v1/train*/{p}.npz")) for p in train_patients}
val_dir = PRIVATE / "export_v1/val"
val_patients = accepted_patients(split["roles"]["val"]["patients"], val_dir)
paths.update({p: val_dir / f"{p}.npz" for p in val_patients})
assert not set(paths) & set(split["roles"]["test"]["patients"]), "A test patient reached this run: stop and report"
assert not set(train_patients) & set(val_patients), "A patient is in training and validation: stop and report"
paths = local_copies(paths, "/content/stage5_exports")  # from here on, reading the data does not need Drive


def load_grid(patient):
    with np.load(paths[patient]) as data:
        arrays = {key: data[key] for key in data.files}
    crop, grid = to_working_grid(arrays)
    grid["post_support"] = crop_forward(arrays["post_support"].astype(np.uint8), crop).astype(bool)
    return grid


def preview_group(group_cache, items, exports, positions, names, categories):
    targets = [{"post_image": exports[p]["post_image"][k], "pre_image": exports[p]["pre_image"][k],
                "label": exports[p]["label"][k], "region": exports[p]["pre_foreground"][k] & exports[p]["post_support"][k],
                "brain": exports[p]["pre_foreground"][k]} for p, k in positions]
    pre_labels = np.stack([exports[p]["pre_label"][k] for p, k in positions])
    return {"cache": group_cache, "items": items, "targets": targets, "names": names, "categories": categories,
            "groups": [p for p, _ in items], "start_image": image_start_latents(group_cache, items, "pre"),
            "start_mask": mask_latents(vae, pre_labels, mask_codec=CODEC, device="cuda")}


encode_text = clip_encoder(tokenizer, text_encoder, "cuda")
started = time.perf_counter()
volumes = {}
for patient in train_patients:
    with np.load(paths[patient]) as data:
        volumes[patient] = label_volumes(data["label"])
ranked = ranked_patients(volumes)
selection, picked, skipped = select_overfit_set(ranked, load_grid, positive=S4["patients"]["positive"],
                                                negative=S4["patients"]["negative"], **S4["slices_per_patient"])
train_exports = {e["patient"]: restrict(picked[e["patient"]], [z for z, _ in e["slices"]]) for e in selection}
del picked
letters = {e["patient"]: "ABCD"[n] for n, e in enumerate(selection)}
val_letters = {p: f"V{n + 1}" for n, p in enumerate(val_patients)}
cache, pools = training_cache(train_patients, load_grid, vae, encode_text, config["text"], mask_codec=CODEC,
                              batch=16, device="cuda")
val_selection, val_exports = {}, {}
for patient in val_patients:
    grid = load_grid(patient)
    val_selection[patient] = select_preview_slices(grid["pre_label"], grid["pre_foreground"],
                                                   **PREVIEWS["validation_slices_per_patient"])
    val_exports[patient] = restrict(grid, [z for z, _ in val_selection[patient]])
    del grid
val_cache = build_feature_cache(val_exports, vae, encode_text, config["text"], batch=8, device="cuda", mask_codec=CODEC)

train_positions = [(e["patient"], k) for e in selection for k in range(len(e["slices"]))]
train_items = [(e["patient"], z) for e in selection for z, _ in e["slices"]]
train_categories = [c for e in selection for _, c in e["slices"]]
train_names = [f"{letters[p]}-{c}-{k + 1}" for (p, k), c in zip(train_positions, train_categories)]
val_items = [(p, k) for p in val_patients for k in range(len(val_selection[p]))]
val_categories = [c for p in val_patients for _, c in val_selection[p]]
val_names = [f"{val_letters[p]}-{c}-{k + 1}" for (p, k), c in zip(val_items, val_categories)]
SETS = {
    "train": preview_group(cache, train_items, train_exports, train_positions, train_names, train_categories),
    "val": preview_group(val_cache, val_items, val_exports, val_items, val_names, val_categories),
}
data_summary = {
    "training_patients": len(train_patients),
    "training_patients_with_id3": sum(bool(pools[p]["positive"]) for p in train_patients),
    "training_slices": sum(len(pools[p]["positive"]) + len(pools[p]["negative"]) for p in train_patients),
    "training_id3_slices": sum(len(pools[p]["positive"]) for p in train_patients),
    "training_prompts": len(cache["text"]),
    "validation_assigned": len(split["roles"]["val"]["patients"]),
    "validation_accepted": len(val_patients),
    "preview_slices": {"train": len(train_items), "val": len(val_items)},
}
record.update(data_summary=data_summary, skipped=[dict(s, rank=ranked[s["kind"]].index(s["patient"]) + 1) for s in skipped],
              cache_seconds=time.perf_counter() - started,
              selection=[{"letter": letters[e["patient"]], "patient": e["patient"], "kind": e["kind"],
                          "slices": [{"frame_z": z, "category": c} for z, c in e["slices"]]} for e in selection],
              val_selection=[{"letter": val_letters[p], "patient": p,
                              "slices": [{"frame_z": z, "category": c} for z, c in val_selection[p]]} for p in val_patients])
print(data_summary, f"| caches took {record['cache_seconds']:.0f}s")
print("skipped (rank, kind, reason):", [(s["rank"], s["kind"], s["reason"]) for s in record["skipped"]])

## 5. Preview helper (fixed-noise loss, generations, scores, PNG pages with the prompts)

In [ ]:
from rhuh_synthesis.overfit import (fixed_eval_loss, follows_source, generate, prompt_caption, save_overfit_pages,
                                   score_slices, shifted_conditions, summarize)
from rhuh_synthesis.pilot import save_stack

GENERATION = dict(num_inference_steps=config["sampling"]["num_inference_steps"],
                  guidance_scale=config["sampling"]["guidance_scale"], seed=PREVIEWS["seed"],
                  batch=PREVIEWS["batch_pairs"], mask_codec=CODEC)
PIXEL_MM = config["representation"]["crop_mm"] / config["representation"]["out_px"]
SHOWN = ("image_mae_mean", "copy_mae_mean", "slice_mae_mean", "id2_dice_mean", "id3_dice_mean", "id3_overlapped", "id3_slices",
         "label_free_labeled_mm2_mean", "invented_id1_mm2_mean", "patient_top1", "mask_patient_top1")


def brief(summary):
    return {k: (round(summary[k], 4) if isinstance(summary[k], float) else summary[k]) for k in SHOWN}


def scored(group, images, masks, step, name):
    rows = score_slices(images, masks, group["targets"], pixel_mm=PIXEL_MM,
                        foreground_threshold=DIAG["metrics"]["foreground_threshold"])
    panels = [{"name": n, "caption": prompt_caption(group["cache"], item), "pre_image": t["pre_image"],
               "post_image": t["post_image"], "generated_image": i, "label": t["label"], "generated_mask": m}
              for n, item, t, i, m in zip(group["names"], group["items"], group["targets"], images, masks)]
    stem = f"step{step:06d}_{name}"
    save_overfit_pages(LOCAL_RUN / "grids" / f"{stem}.png", panels, title=f"Stage 5 {RUN_TAG} step {step} {name}")
    save_stack(LOCAL_RUN / "previews" / f"{stem}.npz", images, masks)
    return {"slices": [dict(row, name=n) for row, n in zip(rows, group["names"])],
            "summary": summarize(rows, group["categories"], groups=group["groups"])}


def preview(unet, step, wrong_patient=False):
    started = time.perf_counter()
    result = {"step": step, "groups": {}}
    for set_name, group in SETS.items():
        entry = {"eval_loss": fixed_eval_loss(unet, group["cache"], group["items"], noise_scheduler,
                                              timesteps=PREVIEWS["eval_loss_timesteps"],
                                              seed=PREVIEWS["eval_loss_noise_seed"], batch=PREVIEWS["eval_loss_batch"])}
        for sampler in PREVIEWS["samplers"]:
            starts = {"start_image": group["start_image"], "start_mask": group["start_mask"]} if sampler == "pair_preop" else {}
            entry[sampler] = scored(group, *generate(unet, vae, ddim, group["cache"], group["items"], **starts, **GENERATION),
                                    step, f"{set_name}_{sampler}")
            print(f"step {step} {set_name} {sampler}:", brief(entry[sampler]["summary"]))
        print(f"step {step} {set_name} loss: image {entry['eval_loss']['image']['mean']:.4f}, "
              f"mask {entry['eval_loss']['mask']['mean']:.4f}")
        result["groups"][set_name] = entry
    if wrong_patient:
        group = SETS["val"]
        wrong_items = shifted_conditions(group["items"], val_patients)
        position = {item: k for k, item in enumerate(group["items"])}
        donors = [position[item] for item in wrong_items]
        control = scored(group, *generate(unet, vae, ddim, group["cache"], group["items"], condition_items=wrong_items,
                                          start_image=group["start_image"][donors], start_mask=group["start_mask"][donors],
                                          **GENERATION), step, "val_wrong_patient")
        control["follows_source"] = follows_source(control["slices"], wrong_items, group["items"])
        result["groups"]["val"]["wrong_patient_preop_pair"] = control
        print(f"step {step} val wrong patient:", brief(control["summary"]),
              f"| nearest to the donor's slice: {control['follows_source']}/{len(wrong_items)}")
    result["seconds"] = time.perf_counter() - started
    print(f"step {step} previews took {result['seconds']:.0f}s")
    return result

## 6. Step 0 previews, then train 1,000 steps with previews at 500 and 1000 (about 50 minutes)

In [ ]:
from rhuh_synthesis.model import expand_conv_in
from rhuh_synthesis.pilot import mirror
from rhuh_synthesis.training import SliceSampler, train


def to_drive():
    report = mirror(LOCAL_RUN, RUN)
    print("copy to Drive:", report if report["error"] is None else f"{report} (training continues, the copy is retried later)")
    return report


LOCAL_RUN.mkdir(parents=True)
unet11 = expand_conv_in(unet4, 11).to("cuda")
del unet4
torch.cuda.empty_cache()
optimizer = torch.optim.AdamW(unet11.parameters(), lr=config["training"]["learning_rate"],
                              weight_decay=config["training"]["weight_decay"])
TRAIN_SEED = config["training"]["train_seed"]
sampler = SliceSampler(pools, seed=TRAIN_SEED, positive_pool_probability=config["training"]["positive_pool_probability"])
rng, generator = np.random.default_rng(TRAIN_SEED), torch.Generator().manual_seed(TRAIN_SEED)
log_path = LOCAL_RUN / "train_log.jsonl"
CHECKPOINTS = Path(f"/content/stage5_{RUN_TAG}_checkpoints")  # full resume checkpoints, about 11 GB each, local only
torch.cuda.reset_peak_memory_stats()
evaluations = [preview(unet11, 0)]
to_drive()
step, train_seconds = 0, 0.0
for target in PREVIEWS["at_steps"][1:]:
    start = time.perf_counter()
    step = train(unet11, cache, sampler, noise_scheduler, optimizer, steps=target - step,
                 accumulation=TRAIN["accumulation"], microbatch=TRAIN["microbatch_pairs"], rng=rng, generator=generator,
                 log_path=log_path, checkpoint_dir=CHECKPOINTS, checkpoint_every=config["training"]["checkpoint_every"],
                 start_step=step, precision=TRAIN["precision"], config_sha256=record["config_sha256"])
    train_seconds += time.perf_counter() - start
    print(f"trained to step {step} ({train_seconds:.0f}s of training so far)")
    evaluations.append(preview(unet11, step, wrong_patient=target == TRAIN["steps"]))
    to_drive()
assert step == TRAIN["steps"]
record["train_seconds"] = train_seconds
record["peak_gib"] = torch.cuda.max_memory_allocated() / 2**30
record["adapter_added_weight_abs_mean"] = float(unet11.conv_in.weight[:, 4:].detach().abs().mean())

## 7. Save the step-1000 UNet for part 2, write-once record and the part to send

In [ ]:
from rhuh_synthesis.manifest import write_once_json

weights = LOCAL_RUN / f"unet_step{step:06d}.pt"
torch.save(unet11.state_dict(), weights)
record["weights"] = {"file": weights.name, "sha256": file_sha256(weights), "gib": weights.stat().st_size / 2**30}
print("saved", weights, record["weights"]["sha256"][:12], f"{record['weights']['gib']:.1f} GiB")
training_log = [json.loads(line) for line in log_path.read_text(encoding="utf-8").splitlines()]
counts = sampler.state_dict()["counts"]
record.update(evaluations=evaluations, training_log=training_log, sampler_counts=counts,
              local_checkpoints=sorted(p.name for p in CHECKPOINTS.glob("*.pt")),
              total_seconds=time.perf_counter() - STARTED)
path, digest = write_once_json(LOCAL_RUN / "record.json", record)
print("saved", path, digest[:12])
losses = [line["loss"] for line in training_log]
shareable = {
    "commit": commit, "config_sha256": record["config_sha256"][:12], "gpu": record["gpu"], "torch": record["torch"],
    "checkpoint_sha256": record["checkpoint"]["sha256"][:12], "weights_sha256": record["weights"]["sha256"][:12],
    "equivalence_tf32_off": record["equivalence_tf32_off"], "equivalence_tf32_on": record["equivalence_tf32_on"],
    "data": data_summary, "skipped": [(s["rank"], s["kind"], s["reason"]) for s in record["skipped"]],
    "sampler_draws": {"positive": sum(c["positive"] for c in counts.values()), "negative": sum(c["negative"] for c in counts.values())},
    "train_loss": {"mean_first_50": float(np.mean(losses[:50])), "mean_at_500": float(np.mean(losses[450:500])),
                   "mean_last_50": float(np.mean(losses[-50:])), "all_finite": all(line["finite"] for line in training_log)},
    "evaluations": [{"step": e["step"], "seconds": round(e["seconds"]),
                     "groups": {name: {key: (value if key == "eval_loss" else {"summary": value["summary"],
                                                                               **({"follows_source": value["follows_source"]}
                                                                                  if "follows_source" in value else {})})
                                       for key, value in group.items()} for name, group in e["groups"].items()}}
                    for e in evaluations],
    "last_step_slices": {name: {key: value["slices"] for key, value in group.items() if key != "eval_loss"}
                         for name, group in evaluations[-1]["groups"].items()},
    "adapter_added_weight_abs_mean": record["adapter_added_weight_abs_mean"],
    "cache_seconds": round(record["cache_seconds"]), "train_seconds": round(record["train_seconds"]),
    "total_seconds": round(record["total_seconds"]), "peak_gib": round(record["peak_gib"], 1),
}
print(json.dumps(shareable, indent=1))

## 8. Finish the upload and confirm every file is on Drive (a few minutes)

Part 2 reads the UNet and the record from Drive. If this cell fails, run it again: the files stay on local disk while this session is open. Colab may ask you to approve the reconnection.

In [ ]:
from rhuh_synthesis.pilot import ensure_on_drive

print(ensure_on_drive(LOCAL_RUN, RUN, drive))
print("Confirmed on Drive:", RUN)

## Next

Save this notebook with its outputs under a name Git ignores (for example `RHUH_GBM_Colab_stage5_trainipynb.ipynb`), or send Claude the printed JSON block; it has slice names (A to D, V1 to V3) but no patient IDs.

Then run part 2, `RHUH_GBM_Colab_stage5_eval.ipynb`, with the same `RUN_TAG`, unless the loss went non-finite or the training stopped with an error. It scores the same step-1000 weights whatever the previews look like, so no choice is made from the previews. It can run in the same session or a new one, because it reads everything from Drive.